# Data Exploration: ArXiv Papers Dataset


### Extract text from pdf

In [9]:
import pymupdf

def extract_text_from_pdf(pdf_path):
    text = ""
    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

### Global Cleaning

In [10]:
import re

def clean_text_with_regex(text):
    # Fix hyphenation: 
    text = re.sub(r'(\w+)-\n(\w+)', r'\1 \2', text)

    # remove known repeating headers/footers
    text = re.sub(r'Published as a conference paper at [^\n]+\n', '', text)
    text = re.sub(r'arXiv:[^\n]+\n', '', text)

    # 3. REMOVE: Non-printable control characters (critical for figure junk)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    
    # 4. CLEANUP: Aggressive removal of non-text symbols around figures/equations
    # Targets single/multi-line strings of symbols that are NOT part of normal text/citations
    text = re.sub(r'[!@#$%^&*()_+={}\[\]|\\:;"\'<,>.?/`~]\n', '\n', text)

    # Remove excessive whitespace and newlines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n\n', text)

    
    text = re.sub(r'\n\s*\d{1,4}\s*\n', '\n\n', text)

    # Remove page numbers (assuming they are on lines by themselves)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)



    return text.strip()



In [11]:

import pdfplumber
def extract_text_with_pdfplumber(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

text_sample_plumber = extract_text_with_pdfplumber("../../data/raw/pdfs/1107.4573v1.pdf")
cleaned_text_plumber = clean_text_with_regex(text_sample_plumber)
cleaned_text_plumber

'1102\nluJ\n\n]IA.sc\n1v3754.7011:viXra\nAnalogy Perception Applied to Seven Tests of\nWord Comprehension\nPeter D. Turney\nInstitute for Information Technology\nNational Research Council of Canada\nM-50 Montreal Road\nOttawa, Ontario, Canada\nK1A 0R6\nPhone: (613) 993-8564\nFax: (613) 952-7151\npeter.turney@nrc-cnrc.gc.ca\nFebruary 23, 2009\nAnalogy Perception Applied to Seven Tests of\nWord Comprehension\nAbstract\nIthasbeenarguedthatanalogyisthecoreofcognition. InAIresearch\nalgorithms for analogy are often limited by the need for hand-coded high level representations as input. An alternative approach is to use high-level\nperception,in which high-levelrepresentationsare automaticallygenerated\nfromraw data. Analogyperceptionis the processof recognizinganalogies\nusing high-level perception. We present PairClass, an algorithm for anal ogyperceptionthatrecognizeslexicalproportionalanalogiesusingrepresen-\ntations that are automatically generated from a large corpus of raw textual\nda

In [12]:
print(cleaned_text_plumber)

1102
luJ

]IA.sc
1v3754.7011:viXra
Analogy Perception Applied to Seven Tests of
Word Comprehension
Peter D. Turney
Institute for Information Technology
National Research Council of Canada
M-50 Montreal Road
Ottawa, Ontario, Canada
K1A 0R6
Phone: (613) 993-8564
Fax: (613) 952-7151
peter.turney@nrc-cnrc.gc.ca
February 23, 2009
Analogy Perception Applied to Seven Tests of
Word Comprehension
Abstract
Ithasbeenarguedthatanalogyisthecoreofcognition. InAIresearch
algorithms for analogy are often limited by the need for hand-coded high level representations as input. An alternative approach is to use high-level
perception,in which high-levelrepresentationsare automaticallygenerated
fromraw data. Analogyperceptionis the processof recognizinganalogies
using high-level perception. We present PairClass, an algorithm for anal ogyperceptionthatrecognizeslexicalproportionalanalogiesusingrepresen-
tations that are automatically generated from a large corpus of raw textual
data. Aproportionalanalogyisa

### Using only metadata for initial exploration

In [13]:
# Load your clean abstract data
import json

with open('../../data/raw/arxiv_metadata.json', 'r', encoding='utf-8') as f:
    papers = json.load(f)

# Explore abstract statistics
abstract_lengths = [len(paper['summary'].split()) for paper in papers]
print(f"Total papers: {len(papers)}")
print(f"Avg abstract length: {sum(abstract_lengths)/len(abstract_lengths):.0f} words")
print(f"Shortest: {min(abstract_lengths)} words")
print(f"Longest: {max(abstract_lengths)} words")

# Sample a clean abstract
print("\nSample abstract:")
print(papers[0]['title'])
print(papers[0]['summary'][:500])

Total papers: 993
Avg abstract length: 168 words
Shortest: 45 words
Longest: 317 words

Sample abstract:
I like fish, especially dolphins: Addressing Contradictions in Dialogue Modeling
To quantify how well natural language understanding models can capture consistency in a general conversation, we introduce the DialoguE COntradiction DEtection task (DECODE) and a new conversational dataset containing both human-human and human-bot contradictory dialogues. We then compare a structured utterance-based approach of using pre-trained Transformer models for contradiction detection with the typical unstructured approach. Results reveal that: (i) our newly collected dataset is notably 


In [14]:
import json
from collections import Counter
from datetime import datetime

with open('../../data/raw/arxiv_metadata.json', 'r', encoding='utf-8') as f:
    papers = json.load(f)

# Analysis
print(f"Total papers: {len(papers)}")

years = [p['published'][:4] for p in papers]
year_dist = Counter(years)
print("Papers per year:")
for year in sorted(year_dist.keys(), reverse=True)[:10]:
    print(f"  {year}: {year_dist[year]}")


Total papers: 993
Papers per year:
  2024: 148
  2023: 309
  2022: 163
  2021: 132
  2020: 99
  2019: 72
  2018: 43
  2017: 7
  2016: 7
  2015: 6


In [15]:
all_authors = []
for p in papers:
    all_authors.extend(p['authors'])
author_dist = Counter(all_authors)
print("\nTop 10 authors by number of papers:")
for author, count in author_dist.most_common(10):
    print(f"  {author}: {count}")


Top 10 authors by number of papers:
  Mohit Bansal: 12
  Ningyu Zhang: 11
  Jianfeng Gao: 10
  Huajun Chen: 10
  Kai-Wei Chang: 9
  Luke Zettlemoyer: 9
  Shumin Deng: 8
  Mikel Artetxe: 8
  Eduard Hovy: 7
  Caiming Xiong: 7


In [16]:
# Abstract length distribution
abstract_lengths = [len(p['summary'].split()) for p in papers]
print(f"\nAbstract length: min={min(abstract_lengths)}, max={max(abstract_lengths)}, avg={sum(abstract_lengths)/len(abstract_lengths):.0f} words")


Abstract length: min=45, max=317, avg=168 words


In [17]:
# sample some titles to see variety of topics
print("\nSample paper titles:")
import random
for p in random.sample(papers, 10):
    print(f" - {p['title']}")


Sample paper titles:
 - Adapting Sequence to Sequence models for Text Normalization in Social Media
 - Re-conceptualising the Language Game Paradigm in the Framework of Multi-Agent Reinforcement Learning
 - Multilingual Alzheimer's Dementia Recognition through Spontaneous Speech: a Signal Processing Grand Challenge
 - To Repeat or Not To Repeat: Insights from Scaling LLM under Token-Crisis
 - Is Model Collapse Inevitable? Breaking the Curse of Recursion by Accumulating Real and Synthetic Data
 - Learning Knowledge Graph-based World Models of Textual Environments
 - LLaMA-Adapter: Efficient Fine-tuning of Language Models with Zero-init Attention
 - Learning to Generate Better Than Your LLM
 - A Comprehensive Survey of AI-Generated Content (AIGC): A History of Generative AI from GAN to ChatGPT
 - VideoDubber: Machine Translation with Speech-Aware Length Control for Video Dubbing


In [18]:
surveys = [p for p in papers if any(word in p['title'].lower() 
           for word in ['survey', 'review', 'tutorial', 'overview'])]
print(f"Found {len(surveys)} potential surveys")
for p in surveys[:20]:
    print(f"  - {p['title']}")

Found 33 potential surveys
  - State Space Model for New-Generation Network Alternative to Transformers: A Survey
  - Navigating the Landscape of Large Language Models: A Comprehensive Review and Analysis of Paradigms and Fine-Tuning Strategies
  - Knowledge Graph Embedding: An Overview
  - Survey on English Entity Linking on Wikidata
  - A Comprehensive Survey of AI-Generated Content (AIGC): A History of Generative AI from GAN to ChatGPT
  - Augmenting LLMs with Knowledge: A survey on hallucination prevention
  - Automatically Correcting Large Language Models: Surveying the landscape of diverse self-correction strategies
  - Explainability for Large Language Models: A Survey
  - Automatic Quality Assessment of Wikipedia Articles -- A Systematic Literature Review
  - An Overview Of Temporal Commonsense Reasoning and Acquisition
  - Testing the effectiveness of saliency-based explainability in NLP using randomized survey-based experiments
  - Reinforcement Learning and Bandits for Speec